In [1]:
# 1. Gỡ cài đặt các gói cũ để tránh xung đột phiên bản chéo
!pip uninstall -y unsloth unsloth_zoo trl transformers datasets peft accelerate bitsandbytes torchao

# 2. Cài đặt đồng bộ tất cả các gói với khoảng phiên bản tương thích
!pip install --no-cache-dir \
    "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" \
    unsloth_zoo \
    msgspec \
    tyro \
    hf_transfer \
    "torchao>=0.13.0" \
    "datasets>=3.4.1,<4.4.0" \
    "transformers>=4.51.3,<=5.5.0" \
    "trl>=0.18.2,<=0.24.0" \
    peft \
    accelerate \
    bitsandbytes

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: datasets 4.8.5
Uninstalling datasets-4.8.5:
  Successfully uninstalled datasets-4.8.5
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-s03tfojq/unsloth_744138522ea04d00858dd3c987b2ff65
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-s03tfojq/unsloth_744138522ea04d00858dd3c987b2ff65
  Resolved https://github.com/unslothai/unsloth.git to commit 6bf101107a7b1fd599d2a829241d93d95488bf94
  Installing build depende

In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/fthetoan/chatbot/chat_dataset.jsonl


In [3]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

max_seq_length = 2048 # Độ dài ngữ cảnh tối đa
dtype = None # Tự động phát hiện (Float16 cho T4)
load_in_4bit = True # Sử dụng lượng tử hóa 4-bit để tiết kiệm VRAM

# 1. Tải Base Model và Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Hoặc "unsloth/qwen2.5-7b-instruct-bnb-4bit"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Cấu hình tham số LoRA (Fine-tuning Parameter)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank của LoRA (8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Tối ưu hóa cho 0
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Tối ưu bộ nhớ cho seq dài
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 3. Chuẩn bị Format Dataset (ChatML)
def format_prompts(examples):
    messages = examples["messages"]
    texts = []
    for msg in messages:
        # Sử dụng tokenizer apply_chat_template để sinh cấu trúc prompt chuẩn
        text = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

# Kiểm tra đường dẫn chính xác của file dataset trong Kaggle (chạy đoạn này nếu gặp FileNotFoundError)
# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# Upload file dataset.jsonl lên Kaggle và load vào (thay thế đường dẫn chính xác in ra ở trên nếu cần)
dataset = load_dataset("json", data_files="/kaggle/input/datasets/fthetoan/chatbot/chat_dataset.jsonl", split="train")
dataset = dataset.map(format_prompts, batched=True)

# 4. Cấu hình các tham số huấn luyện (Hyperparameters)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Có thể tăng tốc cho seq ngắn
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Tăng lên 100-300 tùy thuộc vào kích thước dữ liệu của bạn
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 5. Bắt đầu Huấn luyện
trainer_stats = trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 41 | Num Epochs = 10 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.343390
2,2.185336
3,2.438196
4,2.158781
5,2.084010
6,2.354261
7,1.936097
8,1.917657
9,1.649036
10,1.384504


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [4]:
# 1. Dọn dẹp sạch sẽ ổ đĩa khỏi các file rác bị ghi dở từ lần chạy trước để lấy lại 100% dung lượng trống
!rm -rf /kaggle/working/*gguf /kaggle/working/retech_model_q4 /kaggle/working/outputs

import os
import shutil
import glob

# 2. Lưu thư mục làm việc cũ và chuyển thư mục làm việc hiện tại sang /tmp để tránh tràn đĩa
old_cwd = os.getcwd()
os.chdir("/tmp")

# 3. Lưu mô hình và GGUF vào thư mục tạm /tmp
model.save_pretrained_gguf(
    "retech_model_q4", 
    tokenizer, 
    quantization_method = "q4_k_m",
    temporary_location = "/tmp"
)

# 4. Trở lại thư mục làm việc cũ
os.chdir(old_cwd)

# 5. Sao chép file GGUF cuối cùng từ /tmp về thư mục làm việc /kaggle/working/
gguf_files = glob.glob("/tmp/*retech_model_q4*.gguf") + glob.glob("/tmp/*.gguf")
for f in gguf_files:
    dest = os.path.join("/kaggle/working/", os.path.basename(f))
    print(f"Sao chép file GGUF thành công: {f} -> {dest}")
    shutil.copy(f, dest)

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in retech_model_q4/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:26<00:00, 21.56s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:05<00:00, 46.41s/it]


Unsloth: Merge process complete. Saved to `/tmp/retech_model_q4`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['retech_model_q4_gguf/llama-3-8b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model fil

In [5]:
!cp -r /tmp/retech_model_q4_gguf /kaggle/working/

In [6]:
import requests
import json

# 1. Lấy máy chủ upload trống của Gofile
try:
    print("Đang kết nối đến Gofile API...")
    server_resp = requests.get("https://api.gofile.io/getServer").json()
    if server_resp["status"] == "ok":
        server = server_resp["data"]["server"]
        print(f"-> Đã kết nối thành công. Server nhận file: {server}")
        
        # 2. Tiến hành upload file .gguf
        file_path = "/kaggle/working/retech_model_q4_gguf/llama-3-8b-instruct.Q4_K_M.gguf"
        url = f"https://{server}.gofile.io/uploadFile"
        
        print("-> Đang upload file (4.6GB) lên Gofile. Quá trình này mất khoảng 1-2 phút, vui lòng đợi...")
        with open(file_path, 'rb') as f:
            files = {'file': f}
            upload_resp = requests.post(url, files=files).json()
            
        if upload_resp["status"] == "ok":
            download_page = upload_resp["data"]["downloadPage"]
            print("\n🎉 UPLOAD THÀNH CÔNG!")
            print(f"🔗 Link tải file .gguf của bạn: {download_page}")
        else:
            print(f"❌ Lỗi khi upload từ Gofile: {upload_resp}")
    else:
        print("❌ Không thể lấy máy chủ upload từ Gofile API.")
except Exception as e:
    print(f"❌ Đã xảy ra lỗi kết nối: {e}")

Đang kết nối đến Gofile API...
❌ Đã xảy ra lỗi kết nối: Expecting value: line 1 column 1 (char 0)
